# PEFT Fine-Tuning: Dialogue Summarization with LoRA
Fine-tunes `google/flan-t5-base` on the `knkarthick/dialogsum` dataset using LoRA adapters.

> **Tip:** Go to `Runtime → Change runtime type` and select **GPU** (T4) for faster training.

In [ ]:
# Install dependencies (torchao>=0.16.0 required by recent transformers)
!pip install -q "torchao>=0.16.0" transformers datasets peft accelerate

In [ ]:
import time
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model, TaskType

## 1. Detect Device

In [ ]:
def get_device():
    print('🚀 Initializing the LLM Pipeline...')
    print('--------------------------------------------------')
    print('✅ All required libraries imported successfully.')
    print(f'✅ PyTorch version: {torch.__version__}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'✅ Processing device set to: {device.upper()}')
    print('--------------------------------------------------')
    return device

device = get_device()

## 2. Load Dataset

In [ ]:
def load_dialogue_dataset(dataset_name='knkarthick/dialogsum'):
    print('\n📥 Fetching data from Hugging Face...')
    dataset = load_dataset(dataset_name)
    print('✅ Dataset loaded successfully!')
    print(f'📊 Dataset Structure:\n{dataset}')
    print('\n🔍 Sample Data (Dialogue):')
    print('--------------------------------------------------')
    print(dataset['test'][0]['dialogue'])
    print('--------------------------------------------------')
    print('📝 Human Summary:')
    print(dataset['test'][0]['summary'])
    print('--------------------------------------------------')
    return dataset

dataset = load_dialogue_dataset()

## 3. Load Model & Tokenizer

In [ ]:
def print_number_of_trainable_model_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(
        f'📊 Model Parameter Report:\n'
        f'--------------------------------------------------\n'
        f'  Total Parameters:     {total:,}\n'
        f'  Trainable Parameters: {trainable:,}\n'
        f'  Percentage Trainable: {100 * trainable / total:.4f}%\n'
        f'--------------------------------------------------'
    )

def load_llm_model(device, model_name='google/flan-t5-base'):
    print('\n🧠 Loading the LLM and Tokenizer...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    print('✅ Tokenizer loaded.')
    target_dtype = torch.bfloat16 if device == 'cuda' else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=target_dtype).to(device)
    print(f'✅ {model_name} loaded with dtype={target_dtype}.')
    print_number_of_trainable_model_parameters(model)
    return model, tokenizer

original_model, tokenizer = load_llm_model(device)

## 4. Preprocess & Subsample Dataset

In [ ]:
def preprocess_dataset(tokenizer, dataset):
    print('\n⚙️ Preprocessing and Tokenizing the Dataset...')

    def tokenize_function(example):
        start_prompt = 'Summarize the following dialogue.\n\n'
        end_prompt = '\n\nSummary: '
        safe_dialogues = [str(d) if d is not None else '' for d in example['dialogue']]
        safe_summaries = [str(s) if s is not None else '' for s in example['summary']]
        prompt = [start_prompt + d + end_prompt for d in safe_dialogues]
        example['input_ids'] = tokenizer(prompt, padding='max_length', truncation=True, return_tensors='pt').input_ids
        example['labels'] = tokenizer(safe_summaries, padding='max_length', truncation=True, return_tensors='pt').input_ids
        return example

    tokenized = dataset.map(tokenize_function, batched=True)
    tokenized = tokenized.remove_columns(['id', 'dialogue', 'summary', 'topic'])
    print('✅ Dataset tokenized and cleaned.')
    return tokenized

def subsample_dataset(tokenized_datasets):
    print('\n✂️ Subsampling dataset (1 in every 100 records)...')
    small = tokenized_datasets.filter(lambda example, idx: idx % 100 == 0, with_indices=True)
    print('✅ Dataset subsampled.')
    print(f'  Train:      {small["train"].shape}')
    print(f'  Validation: {small["validation"].shape}')
    print(f'  Test:       {small["test"].shape}')
    return small

tokenized_dataset = preprocess_dataset(tokenizer, dataset)
small_tokenized_dataset = subsample_dataset(tokenized_dataset)

## 5. Inject LoRA Adapters

In [ ]:
def setup_peft_lora_model(original_model):
    print('\n🪄 Injecting LoRA Adapters into the Model (PEFT)...')
    lora_config = LoraConfig(
        r=32,
        lora_alpha=32,
        target_modules=['q', 'v'],
        lora_dropout=0.05,
        bias='none',
        task_type=TaskType.SEQ_2_SEQ_LM,
    )
    peft_model = get_peft_model(original_model, lora_config)
    print('✅ LoRA adapters injected!')
    print_number_of_trainable_model_parameters(peft_model)
    return peft_model

peft_model = setup_peft_lora_model(original_model)

## 6. Train & Save

> `max_steps=1` is a dry run. Remove it (or increase `num_train_epochs`) for a full training run.

In [ ]:
def train_and_save_peft_model(peft_model, tokenizer, tokenized_datasets):
    print('\n🏋️ Starting PEFT Training...')
    output_dir = f'./peft-dialogue-summary-training-{int(time.time())}'

    training_args = TrainingArguments(
        output_dir=output_dir,
        auto_find_batch_size=True,
        learning_rate=1e-3,
        num_train_epochs=1,
        logging_steps=1,
        max_steps=1,  # Remove for full training
    )

    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=tokenized_datasets['train'],
    )

    print('⏳ Running trainer...')
    trainer.train()

    save_path = './peft-dialogue-summary-checkpoint-local'
    trainer.model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    print(f'✅ PEFT adapters saved to: {save_path}')
    return save_path

adapter_path = train_and_save_peft_model(peft_model, tokenizer, small_tokenized_dataset)
print('\n✅ PEFT training complete! Adapter weights saved to:', adapter_path)